[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philmui/worldmodels/blob/main/gait/skeleton-jepa/gavd2/04-pretrain-jepa-at-scale.ipynb)

# Part 4: Pretrain the Skeleton-JEPA at scale

This is the heart of the series. We take the unlabeled clip bank from notebook 03
and train a Skeleton-JEPA on it, with no labels at all. The model plays a
fill-in-the-blank game millions of times: hide a spatiotemporal block of the
skeleton, predict what the hidden joints were doing in latent space, and improve
by lowering the error. Do this across the whole corpus and the encoder is forced
to learn how walking bodies actually move.

A JEPA has four pieces, and we build them exactly as the concept series does. The
context encoder reads the visible joints and summarizes them. The target encoder
is a slow-moving copy of the context encoder that produces the answer key for the
hidden joints. The predictor guesses that answer key from the context. The loss
measures the error in latent space and adds VICReg terms that stop the model from
cheating with a constant output. Nothing here needs negative pairs or a decoder.

We tokenize each clip into one token per joint per frame, and we give the encoder a
learned time embedding and a learned joint embedding so it knows which frame and which
landmark each token came from. This one detail is what makes the encoder a gait model
rather than a bag of coordinates: without it a transformer over flattened tokens is blind
to frame order and left-right identity, and gait is exactly timing and left-right
structure. We then mask a block and run the loop. Throughout we log the loss terms and the
embedding spread, because the one failure mode to watch for is collapse, where the encoder
outputs nearly the same vector for everything. If VICReg is doing its job, the embedding
standard deviation stays healthy and the model keeps learning real structure.

## Run this locally or in Google Colab

In Colab, click the badge and run top to bottom. On your laptop, from the `gavd2/`
folder, `uv sync` then `uv run jupyter lab 04-pretrain-jepa-at-scale.ipynb`.

With `SMOKE_TEST = True`, we synthesize a small corpus and train for a few quick
steps so the whole loop runs in seconds on CPU. With `SMOKE_TEST = False` (the
value these committed notebooks ship with) we load the real `corpus.npz` from
notebook 03 and train for longer. The model stays small on purpose, a 2-layer
transformer with a 64 dimensional embedding, so the same code runs on a laptop or
a modest GPU.

## The four pieces, wired together

The figure shows the training step: visible tokens flow into the context encoder,
the full sequence flows into the EMA target encoder to make the answer key, the
predictor guesses the hidden targets, and the loss compares them in latent space
while VICReg keeps the embeddings from collapsing.

![The JEPA training step](images/jepa-training.svg)

*Context encoder reads the visible joints, the slow EMA target makes the answer key, the predictor guesses, and the latent loss plus VICReg drives learning without collapse.*

## Colab setup

In [ ]:
# Colab setup and local .env loading.
import importlib.util, subprocess, sys

_import_name = {"scikit-learn": "sklearn", "opencv-python": "cv2",
                "yt-dlp": "yt_dlp", "python-dotenv": "dotenv"}

def _ensure(pkgs):
    """pip install any packages whose import is not already available."""
    missing = [p for p in pkgs if importlib.util.find_spec(_import_name.get(p, p)) is None]
    if missing:
        print("Installing:", " ".join(missing))
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + missing)
    return missing

_ensure(["numpy", "pandas", "matplotlib", "torch", "python-dotenv", "tqdm"])

from dotenv import load_dotenv, find_dotenv
import os
load_dotenv(find_dotenv())
print("Loaded environment via load_dotenv(find_dotenv()).")

GAVD_CACHE_DIR = os.getenv("GAVD_CACHE_DIR")
print("Setup complete.")

## Configuration

In [ ]:
from pathlib import Path
import hashlib, random, numpy as np, torch

CONFIG = {
    "SMOKE_TEST": False,          # True -> tiny synthetic corpus, few steps (default, runs anywhere).
                                  # False -> load the real corpus.npz from notebook 03 and train longer.
    "CACHE_DIR": Path(GAVD_CACHE_DIR) if GAVD_CACHE_DIR else Path.cwd() / "cache",
    "T": 32, "N_JOINTS": 33, "C": 3,
    "EMBED_DIM": 64,
    "EMA_M": 0.996,
    # L2 prediction dominates; variance and covariance are light anti-collapse guards
    # applied to the online representation only (see vicreg_loss for why the earlier
    # heavy, target-side weights made the loss climb over long training). VAR_TARGET
    # is the std the variance hinge pushes each dimension up to; keeping it modest
    # (0.5) makes variance a guard that only acts when the spread drops too low,
    # rather than a constant upward force that fights the prediction loss.
    "VICREG_SIM": 25.0, "VICREG_VAR": 0.5, "VICREG_COV": 0.04,
    "VAR_TARGET": 0.5, "EPS": 1e-4,
    "MASK_RATIO": 0.4,           # rough fraction of tokens to hide per clip
    "BATCH": 16,
    "STEPS": 50,                 # smoke: few steps. Real: raised to 400 below.
    "LR": 1e-3,
    "SEED": 42,
    # When True (and REAL), read from the exploratory first-N cache namespace ("_firstN")
    # so it never mixes with the locked (exp5-exact) cache.
    "EXPLORATORY_FIRST_N": False,
}
# Cache namespace: "" for the locked (exp5-exact) run, "_firstN" for the exploratory run.
# Every artifact filename in the series carries this suffix so the two never mix.
CONFIG["CACHE_NS"] = "_firstN" if CONFIG.get("EXPLORATORY_FIRST_N") else ""
if not CONFIG["SMOKE_TEST"]:
    CONFIG["STEPS"] = 400        # a fuller run on the real corpus

# Shared helpers threaded through every notebook in the series. Repeated here so this
# notebook stays self-contained and can execute without the earlier notebooks in memory.

# Canonicalize condition spellings so the cerebral-palsy class never silently drops.
# The full GAVD tree names the folder "cerebral palsy" (with a space); exp5 and
# iteration 2 use "cerebralpalsy" (no space). We canonicalize the LABEL everywhere;
# nb02 already writes canonical labels into caches, and we canonicalize defensively at
# every read so a stale space-form cache cannot drop a class.
CANONICAL_COND = {"cerebral palsy": "cerebralpalsy"}
def canon_cond(c):
    c = str(c).strip().lower()
    return CANONICAL_COND.get(c, c.replace(" ", ""))

# Deterministic fingerprint of the locked 68 ids, stamped onto every cache artifact so a
# downstream notebook can detect (and warn about) a stale-artifact mix.
def canonical_id_hash(ids):
    return hashlib.sha1("\n".join(sorted(map(str, ids))).encode()).hexdigest()[:12]

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
set_seed(CONFIG["SEED"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device = {device}")
print("CONFIG:", {k: v for k, v in CONFIG.items() if k != "CACHE_DIR"})
print(f"Cache namespace: '{CONFIG['CACHE_NS']}'  (locked run reads un-suffixed artifacts)")

## Helpers

We reuse the same edges, groups, normalization, synthetic generator, and inline
animation as the earlier notebooks, so a clip looks and behaves the same
everywhere.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

EDGES = [
    (0,1),(1,2),(2,3),(0,4),(4,5),(5,6),(0,9),(0,10),(9,10),
    (11,12),(11,23),(12,24),(23,24),
    (11,13),(13,15),(15,17),(15,19),(15,21),(17,19),
    (12,14),(14,16),(16,18),(16,20),(16,22),(18,20),
    (23,25),(25,27),(27,29),(27,31),(29,31),
    (24,26),(26,28),(28,30),(28,32),(30,32),
]
GROUPS = {
    "face":[0,1,2,3,4,5,6,7,8,9,10], "left_arm":[11,13,15,17,19,21],
    "right_arm":[12,14,16,18,20,22], "torso":[11,12,23,24],
    "left_leg":[23,25,27,29,31], "right_leg":[24,26,28,30,32],
}

def normalize_skeleton_seq(seq):
    seq = seq.astype(np.float32).copy()
    hips = (seq[:, 23, :] + seq[:, 24, :]) / 2.0
    shoulders = (seq[:, 11, :] + seq[:, 12, :]) / 2.0
    seq = seq - hips[:, None, :]
    torso = np.linalg.norm(shoulders - hips, axis=1)
    scale = np.median(torso[torso > 1e-6]) if np.any(torso > 1e-6) else 1.0
    return (seq / (scale + 1e-6)).astype(np.float32)

def synthesize_walking_skeleton(T=32, seed=0, gait_bias=0.0):
    rng = np.random.RandomState(seed)
    base = np.zeros((33, 3), dtype=np.float32)
    # Full (x, y) layout for all 33 landmarks, laid out as a person seen head-on.
    # y grows downward (head near 0.16, feet near 0.97). x has the midline at 0.50,
    # with the left side (odd joint indices) left of it and the right side right of it.
    # Shoulders are wider than the hips, the arms hang OUTSIDE the hips down to about
    # hip height, and the head sits just above the shoulders, so the figure reads as a
    # real walking body instead of collapsing onto one vertical line.
    xs = {
        0:0.500,                                            # nose
        1:0.485, 2:0.475, 3:0.465, 4:0.515, 5:0.525, 6:0.535,  # eyes (left then right)
        7:0.455, 8:0.545,                                   # ears
        9:0.485, 10:0.515,                                  # mouth
        11:0.415, 12:0.585,                                 # shoulders (wide)
        13:0.395, 14:0.605,                                 # elbows (arms hang outside)
        15:0.405, 16:0.595,                                 # wrists
        17:0.395, 18:0.605, 19:0.405, 20:0.595, 21:0.420, 22:0.580,  # hands track their wrist
        23:0.455, 24:0.545,                                 # hips (narrower than shoulders)
        25:0.450, 26:0.550,                                 # knees
        27:0.448, 28:0.552,                                 # ankles
        29:0.448, 30:0.552, 31:0.455, 32:0.545,             # heels, foot tips
    }
    ys = {
        0:0.16,                                             # nose
        1:0.145, 2:0.145, 3:0.145, 4:0.145, 5:0.145, 6:0.145,  # eyes
        7:0.155, 8:0.155,                                   # ears
        9:0.185, 10:0.185,                                  # mouth (short neck to shoulders)
        11:0.24, 12:0.24,                                   # shoulders
        13:0.38, 14:0.38,                                   # elbows
        15:0.51, 16:0.51,                                   # wrists (about hip height)
        17:0.545, 18:0.545, 19:0.545, 20:0.545, 21:0.535, 22:0.535,  # hands (just past wrists)
        23:0.50, 24:0.50,                                   # hips
        25:0.71, 26:0.71,                                   # knees
        27:0.92, 28:0.92,                                   # ankles
        29:0.94, 30:0.94, 31:0.965, 32:0.965,               # heels, foot tips
    }
    for j in range(33):
        base[j, 0] = xs[j]
        base[j, 1] = ys[j]
    seq = np.repeat(base[None], T, axis=0)
    t = np.linspace(0, 2*np.pi, T, endpoint=False)
    swing = 0.06 * np.sin(t)
    for k, amp in [(25,1.0),(27,1.3),(31,1.4),(13,-0.8),(15,-1.0)]:
        seq[:, k, 0] += swing * amp * (1.0 + gait_bias)
    for k, amp in [(26,-1.0),(28,-1.3),(32,-1.4),(14,0.8),(16,1.0)]:
        seq[:, k, 0] += swing * amp * (1.0 - gait_bias)
    seq += rng.randn(T, 33, 3).astype(np.float32) * 0.004
    return normalize_skeleton_seq(seq)

def animate_skeleton(seq, edges, title="Walking skeleton", mask=None, fps=8):
    T = seq.shape[0]; x_all = seq[:, :, 0]; y_all = seq[:, :, 1]
    groups = [list(range(11)), [11,13,15,17,19,21], [12,14,16,18,20,22],
              [11,12,23,24], [23,25,27,29,31], [24,26,28,30,32]]
    colors = ["#8b5cf6","#3b82f6","#ef4444","#22c55e","#f59e0b","#ec4899"]; grey="#cbd5e1"
    joint_mask = None
    if mask is not None:
        if mask.shape == (T, len(groups)):
            joint_mask = np.zeros((T, 33), dtype=bool)
            for t in range(T):
                for g_idx, grp in enumerate(groups):
                    if mask[t, g_idx]:
                        joint_mask[t, grp] = True
        else:
            joint_mask = mask
    fig, ax = plt.subplots(figsize=(6, 7))
    x_min, x_max = x_all.min(), x_all.max(); y_min, y_max = y_all.min(), y_all.max()
    margin = max(x_max - x_min, y_max - y_min) * 0.1 + 1e-3
    def draw_frame(t):
        ax.clear(); ax.set_aspect('equal'); ax.invert_yaxis(); ax.axis('off')
        ax.set_xlim(x_min - margin, x_max + margin); ax.set_ylim(y_max + margin, y_min - margin)
        ax.set_title(f"{title} (frame {t}/{T})")
        x = x_all[t]; y = y_all[t]
        for g_idx, grp in enumerate(groups):
            for (i, j) in [(i, j) for (i, j) in edges if i in grp and j in grp]:
                hidden = joint_mask is not None and (joint_mask[t, i] or joint_mask[t, j])
                ax.plot([x[i], x[j]], [y[i], y[j]], color=(grey if hidden else colors[g_idx]),
                        linewidth=2, alpha=0.7)
            cg = [grey if (joint_mask is not None and joint_mask[t, i]) else colors[g_idx] for i in grp]
            ax.scatter([x[i] for i in grp], [y[i] for i in grp], c=cg, s=40,
                       zorder=3, edgecolors='white', linewidths=0.5)
    anim = FuncAnimation(fig, draw_frame, frames=T, interval=1000/fps, repeat=True)
    plt.close(fig)
    return HTML(anim.to_jshtml())

print("Helpers defined.")

## The four JEPA pieces

Here are the pieces themselves, identical in structure to the concept series so
the two stay in lockstep. The context encoder and predictor learn by gradient
descent; the target encoder never gets gradients and only moves by a slow EMA
update, which is the asymmetry that lets JEPA learn without negatives. We also
define the block-masking function that creates the fill-in-the-blank puzzle.

In [ ]:
# The four JEPA pieces, identical in structure to the concept series (tutorials/03),
# with one addition the concept series calls for but leaves out: learned positional
# embeddings. Because we flatten each clip into T*33 tokens, a plain transformer would
# treat those tokens as an unordered set. It could never tell frame 0 from frame 31 or
# a left knee from a right knee, so the pooled embedding would throw away all timing and
# all joint identity. Gait is exactly timing and left/right structure, so we add a learned
# time embedding (one vector per frame) and a learned joint embedding (one vector per
# landmark) and add them to each token. This is the standard fix in I-JEPA and V-JEPA,
# and it is what makes the encoder able to represent motion over time at all.
import copy
import torch
import torch.nn as nn
import torch.nn.functional as F

class ContextEncoder(nn.Module):
    """Maps tokens (B, T*n_joints, C) to embeddings (B, T*n_joints, D) with a small
    transformer, after adding a learned time embedding and a learned joint embedding
    so the model knows which frame and which landmark each token came from.

    Tokens must be in row-major (t, j) order: token index n = t * n_joints + j, which is
    exactly what clips.reshape(B, T * n_joints, C) produces. The encoder rebuilds the
    matching positional grid, so notebook 05 gets identical features as long as it uses
    the same T and n_joints (saved in the checkpoint config) and the same token order."""
    def __init__(self, input_dim=3, embed_dim=64, n_layers=2, n_heads=4, T=32, n_joints=33):
        super().__init__()
        self.T = T
        self.n_joints = n_joints
        self.embed_dim = embed_dim
        self.input_proj = nn.Linear(input_dim, embed_dim)
        # Learned positional embeddings. std 0.1 puts them on the same scale as the
        # projected coordinates, so they actually influence attention rather than being
        # washed out by the transformer's internal LayerNorm.
        self.time_embed = nn.Parameter(torch.randn(T, embed_dim) * 0.1)
        self.joint_embed = nn.Parameter(torch.randn(n_joints, embed_dim) * 0.1)
        layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=n_heads,
            dim_feedforward=embed_dim * 2, batch_first=True, dropout=0.0, activation="gelu")
        self.transformer = nn.TransformerEncoder(layer, num_layers=n_layers)
    def forward(self, x):
        # x: (B, T*n_joints, C) in row-major (t, j) order.
        h = self.input_proj(x)                                                  # (B, N, D)
        pos = self.time_embed[:, None, :] + self.joint_embed[None, :, :]        # (T, n_joints, D)
        pos = pos.reshape(self.T * self.n_joints, self.embed_dim)               # (N, D)
        return self.transformer(h + pos[None, :, :])                            # broadcast over batch

def make_target_encoder(context_encoder):
    target = copy.deepcopy(context_encoder)
    for p in target.parameters():
        p.requires_grad = False
    return target

@torch.no_grad()
def ema_update(target_encoder, context_encoder, m):
    for p_tgt, p_ctx in zip(target_encoder.parameters(), context_encoder.parameters()):
        p_tgt.data.mul_(m).add_(p_ctx.data, alpha=(1.0 - m))

class Predictor(nn.Module):
    """Guesses hidden token embeddings from context + position. (B, N, D_in) -> (B, N, D_out)."""
    def __init__(self, input_dim=64, output_dim=64, hidden_dim=None):
        super().__init__()
        hidden_dim = hidden_dim or input_dim * 2
        self.net = nn.Sequential(nn.Linear(input_dim, hidden_dim), nn.GELU(),
                                 nn.Linear(hidden_dim, output_dim))
    def forward(self, x):
        return self.net(x)

def vicreg_loss(pred, target, cfg, context=None):
    """L2 prediction loss on the (LayerNorm-normalized) target, plus light VICReg
    variance + covariance ON THE ONLINE SIDE ONLY, all over (B, N, D).

    Two details keep the loss from running away over long training, and both match
    how real JEPAs (I-JEPA, V-JEPA) are built:
      1. We LayerNorm the EMA target before the L2. The target encoder is a slow
         copy of the online encoder, so its embedding scale drifts as training goes.
         Normalizing the target means the prediction loss measures direction, not
         magnitude, so a growing target scale can no longer inflate the loss.
      2. The variance and covariance anti-collapse terms are applied only to the
         online representation (the context embedding when given, otherwise the
         prediction), never to the stop-gradient target. Regularizing the target
         through its EMA would push the online encoder to inflate the whole
         representation, which is exactly what makes the loss climb.
    """
    B, N, D = pred.shape
    assert target.shape == pred.shape
    if B < 2:
        raise ValueError("VICReg needs batch size >= 2 for variance/covariance")
    lam_sim, lam_var, lam_cov = cfg["VICREG_SIM"], cfg["VICREG_VAR"], cfg["VICREG_COV"]
    gamma, eps = cfg["VAR_TARGET"], cfg["EPS"]

    # 1. Invariance: L2 between the prediction and the NORMALIZED target.
    tgt_norm = F.layer_norm(target, (D,))
    pf = pred.reshape(-1, D); tf = tgt_norm.reshape(-1, D)
    sim_loss = F.mse_loss(pf, tf)

    # Anti-collapse terms act on the online representation only.
    online = context if context is not None else pred
    of = online.reshape(-1, D)

    def variance_term(x):
        std = torch.sqrt(x.var(dim=0, unbiased=True) + eps)
        return F.relu(gamma - std).mean()

    def covariance_term(x):
        xc = x - x.mean(dim=0, keepdim=True)
        cov = (xc.T @ xc) / max(x.size(0) - 1, 1)
        cov = cov - torch.diag(torch.diag(cov))
        return cov.pow(2).sum() / x.size(1)

    var_loss = variance_term(of)
    cov_loss = covariance_term(of)
    total = lam_sim * sim_loss + lam_var * var_loss + lam_cov * cov_loss
    return total, {"sim": sim_loss.detach().item(), "var": var_loss.detach().item(),
                   "cov": cov_loss.detach().item()}

def make_block_mask(T, groups, style, ratio, rng):
    """Spatiotemporal block mask -> (T, n_groups) bool, True = hidden."""
    names = list(groups.keys()); n_groups = len(names)
    mask = np.zeros((T, n_groups), dtype=bool)
    if style == "limb":
        wl = min(max(1, int(ratio * T * n_groups)), T)
        limbs = [i for i, nm in enumerate(names) if nm in ["left_arm","right_arm","left_leg","right_leg"]]
        gi = rng.choice(limbs if limbs else list(range(n_groups)))
        st = rng.randint(0, max(1, T - wl + 1)); mask[st:min(st+wl, T), gi] = True
    elif style == "time":
        wl = min(max(1, int(ratio * T)), T)
        st = rng.randint(0, max(1, T - wl + 1)); mask[st:min(st+wl, T), :] = True
    else:
        raise ValueError(f"Unknown style: {style}")
    return mask

print("JEPA pieces defined: ContextEncoder (with time+joint positional embeddings), "
      "make_target_encoder, ema_update, Predictor, vicreg_loss, make_block_mask.")

## Load the pretraining corpus

We load the unlabeled clip bank from notebook 03. If it is missing, we synthesize
a small corpus so this notebook still runs on its own. Each clip is a normalized
`(T, 33, 3)` array, and the whole bank is one big `(N, T, 33, 3)` tensor that we
draw random batches from.

In [ ]:
ns = CONFIG["CACHE_NS"]
corpus_path = CONFIG["CACHE_DIR"] / f"corpus{ns}.npz"
holdout_path = CONFIG["CACHE_DIR"] / f"labeled_holdout{ns}.npz"
hash_path = CONFIG["CACHE_DIR"] / f"canonical_id_hash{ns}.txt"

# provenance stamp we will thread into the encoder checkpoint below so notebook 05 can
# detect a mismatched encoder. "smoke-or-firstN" is the fallback string for any path
# where the exp5 lock is not in force (SMOKE, exploratory firstN, or older nb03 caches).
CORPUS_HASH = "smoke-or-firstN"

if not CONFIG["SMOKE_TEST"] and corpus_path.exists():
    data = np.load(corpus_path, allow_pickle=True)
    corpus = data["clips"].astype(np.float32)
    print(f"Loaded real corpus from {corpus_path.name}: {corpus.shape}")

    # Read the corpus's own stamped hash (nb03 iteration-2 writes it). Older nb03
    # builds omit the key; we tolerate that but note it.
    if "canonical_id_hash" in data.files:
        CORPUS_HASH = str(data["canonical_id_hash"])
        if hash_path.exists():
            expected_hash = hash_path.read_text().strip()
            if expected_hash != CORPUS_HASH:
                print(f"WARNING: canonical_id_hash mismatch (stale-artifact mix?). "
                      f"corpus.npz stamped {CORPUS_HASH}, cache/canonical_id_hash{ns}.txt "
                      f"has {expected_hash}. Continuing with the corpus's own hash.")
            else:
                print(f"canonical_id_hash matches ({CORPUS_HASH}). Locked-lineage OK.")
        else:
            print(f"canonical_id_hash{ns}.txt not on disk; using corpus.npz stamp {CORPUS_HASH}.")
    else:
        print("NOTE: corpus.npz has no canonical_id_hash key (older nb03 build). "
              "Proceeding without the exp5-lock provenance stamp; nb05 will see "
              "'smoke-or-firstN' on the encoder checkpoint.")

    # Load the labelled holdout only for the pretrain-on-everything-except-the-68
    # provenance line. We never touch labels or holdout clips in this notebook.
    if holdout_path.exists():
        hdata = np.load(holdout_path, allow_pickle=True)
        # Iteration-2 nb03 writes expected_seq_ids and missing_seq_ids; older builds
        # do not. Missing -> we cannot compute K precisely and print a note instead.
        if "expected_seq_ids" in hdata.files and "missing_seq_ids" in hdata.files:
            expected = list(hdata["expected_seq_ids"])
            missing = list(hdata["missing_seq_ids"])
            K = len(expected) - len(missing)
            print(f"Pretraining on {corpus.shape[0]} unlabeled clips; the labelled probe set "
                  f"has expected labelled {len(expected)} vs survived {K} vs missing {len(missing)} "
                  f"(sequences that extraction lost, not sequences excluded from the corpus).")
        else:
            print(f"labeled_holdout{ns}.npz has no expected_seq_ids/missing_seq_ids "
                  f"(older nb03 build). Skipping the expected-vs-survived-vs-missing line.")
    else:
        print(f"labeled_holdout{ns}.npz not on disk; skipping the labelled-provenance line.")
else:
    if not CONFIG["SMOKE_TEST"]:
        print(f"corpus{ns}.npz not found; run notebook 03 first. Falling back to synthetic.")
    biases = [0.0, 0.2, -0.25, 0.1, -0.1, 0.05]
    corpus = np.stack([synthesize_walking_skeleton(T=CONFIG["T"], seed=i,
                       gait_bias=biases[i % len(biases)]) for i in range(64)], axis=0)
    print(f"SMOKE corpus: {corpus.shape}")

corpus_t = torch.from_numpy(corpus).to(device)
print(f"Corpus tensor: {tuple(corpus_t.shape)} on {device}")
print(f"Provenance hash to stamp on the encoder: {CORPUS_HASH}")

## Watch one training clip

Before training, we play one clip from the corpus with a block mask applied, so
you can see the exact fill-in-the-blank puzzle the model faces. The greyed joints
are hidden; the model only sees the colored ones and must predict what the grey
ones were doing. Hiding a whole limb over time, rather than scattered dots, is
what forces the model to reason about coordinated motion.

In [ ]:
rng_demo = np.random.RandomState(0)
demo_clip = corpus[0]
demo_mask = make_block_mask(CONFIG["T"], GROUPS, style="limb", ratio=CONFIG["MASK_RATIO"], rng=rng_demo)
print("Animating one clip with a limb-over-time block masked out (grey = hidden).")
display(animate_skeleton(demo_clip[:24], EDGES, title="Masked training clip", mask=demo_mask[:24]))

## Tokenize and mask a batch

Each clip becomes `T` times 33 tokens, one per joint per frame, each a 3-vector of
coordinates. We flatten in row-major `(t, j)` order, so token `t * 33 + j` is joint `j`
at frame `t`. That order matters, because the encoder adds a learned time embedding and a
learned joint embedding to each token using exactly this layout, which is how the model
recovers which frame and which landmark a token came from. We build a boolean token mask
per clip from a block mask, mixing the limb-over-time and time-window styles across the
batch. The context encoder sees only the visible tokens; the target encoder sees them all
and provides the answer key at the masked positions. The helper below prepares one batch:
the visible tokens, the full tokens, and the masked-position index.

In [ ]:
def joint_group_index():
    """Map each of the 33 joints to its semantic group index (for limb masks)."""
    idx = np.zeros(33, dtype=int)
    for gi, (_, joints) in enumerate(GROUPS.items()):
        for j in joints:
            idx[j] = gi
    return idx

JOINT_GROUP = joint_group_index()

def make_batch(corpus_t, cfg, rng):
    """Return (tokens, token_mask) where tokens is (B, T*33, C) and token_mask is (B, T*33) bool (True=hidden)."""
    B, T, J, C = cfg["BATCH"], cfg["T"], cfg["N_JOINTS"], cfg["C"]
    n = corpus_t.shape[0]
    sel = rng.randint(0, n, size=B)
    clips = corpus_t[sel]                                  # (B, T, J, C)
    tokens = clips.reshape(B, T * J, C)                    # (B, T*J, C)
    token_mask = np.zeros((B, T, J), dtype=bool)
    for b in range(B):
        style = "limb" if rng.rand() < 0.5 else "time"
        gm = make_block_mask(T, GROUPS, style=style, ratio=cfg["MASK_RATIO"], rng=rng)  # (T, n_groups)
        for t in range(T):
            for j in range(J):
                if gm[t, JOINT_GROUP[j]]:
                    token_mask[b, t, j] = True
    token_mask = token_mask.reshape(B, T * J)
    return tokens, torch.from_numpy(token_mask).to(tokens.device)

# Quick shape check.
_rng = np.random.RandomState(1)
_tok, _m = make_batch(corpus_t, CONFIG, _rng)
print(f"batch tokens: {tuple(_tok.shape)}, mask hidden fraction: {_m.float().mean().item():.2f}")

## The training loop

Each step we build a batch, run the context encoder on the visible tokens and the
target encoder on the full sequence, pool the context into a summary that the
predictor uses to guess the masked target embeddings, and compute the loss on the
masked positions only. We backpropagate into the context encoder and predictor,
then nudge the target encoder with an EMA update. We log the loss terms and the
embedding standard deviation so we can confirm the model is learning without
collapsing.

Two choices in the loss keep it well behaved over a long run, and both match how
real JEPAs are built. First, we normalize the target embedding before the L2, so
the prediction loss measures direction rather than magnitude. Without this, the
target encoder's embedding scale slowly grows as training goes, and the raw L2
would climb along with it even when prediction is fine. Second, we apply the
variance and covariance anti-collapse terms to the online context embedding only,
never to the stop-gradient target, and we keep them light so the L2 leads. Those
two choices are what let the loss fall steadily instead of turning back upward.

In [ ]:
context_encoder = ContextEncoder(input_dim=CONFIG["C"], embed_dim=CONFIG["EMBED_DIM"],
                                 T=CONFIG["T"], n_joints=CONFIG["N_JOINTS"]).to(device)
target_encoder = make_target_encoder(context_encoder)
predictor = Predictor(input_dim=CONFIG["EMBED_DIM"], output_dim=CONFIG["EMBED_DIM"]).to(device)
opt = torch.optim.Adam(list(context_encoder.parameters()) + list(predictor.parameters()), lr=CONFIG["LR"])

rng = np.random.RandomState(CONFIG["SEED"])
history = {"total": [], "sim": [], "var": [], "cov": [], "emb_std": []}

for step in range(CONFIG["STEPS"]):
    tokens, token_mask = make_batch(corpus_t, CONFIG, rng)     # (B, N, C), (B, N)
    B, N, C = tokens.shape

    # Context encoder sees only visible tokens (we zero the hidden ones as a simple mask).
    # The positional embeddings inside the encoder still mark where the holes are, so the
    # model knows which frame and joint each zeroed token stands for.
    visible = tokens.clone()
    visible[token_mask] = 0.0
    ctx = context_encoder(visible)                             # (B, N, D)

    # Target encoder sees the full sequence, no gradients.
    with torch.no_grad():
        tgt_all = target_encoder(tokens)                       # (B, N, D)

    # Predictor: for each token, combine a mean-pooled context summary with that token's
    # own context embedding, then predict its target embedding.
    ctx_summary = ctx.mean(dim=1, keepdim=True).expand(-1, N, -1)   # (B, N, D)
    pred_all = predictor(ctx + ctx_summary)                    # (B, N, D)

    # Loss on masked positions only. Gather them into a (B, k, D) block per clip.
    # We keep k fixed per batch by taking the min count so shapes stay regular.
    k = int(token_mask.sum(dim=1).min().item())
    if k < 2:
        continue                                               # need a couple of masked tokens
    pred_masked, tgt_masked, ctx_masked = [], [], []
    for b in range(B):
        idx = torch.where(token_mask[b])[0][:k]
        pred_masked.append(pred_all[b, idx]); tgt_masked.append(tgt_all[b, idx])
        ctx_masked.append(ctx[b, idx])
    pred_masked = torch.stack(pred_masked, 0)                  # (B, k, D)
    tgt_masked = torch.stack(tgt_masked, 0)
    ctx_masked = torch.stack(ctx_masked, 0)                    # online context embeddings

    # L2 on the normalized target; variance/covariance regularize the online context.
    loss, parts = vicreg_loss(pred_masked, tgt_masked, CONFIG, context=ctx_masked)
    opt.zero_grad(); loss.backward(); opt.step()
    ema_update(target_encoder, context_encoder, CONFIG["EMA_M"])

    # Collapse monitor: std of the online context embeddings (what the variance
    # term regularizes). Healthy is comfortably above 0; collapse drives it to 0.
    emb_std = ctx_masked.reshape(-1, ctx_masked.shape[-1]).std(dim=0).mean().item()
    history["total"].append(loss.item()); history["sim"].append(parts["sim"])
    history["var"].append(parts["var"]); history["cov"].append(parts["cov"])
    history["emb_std"].append(emb_std)
    if step % max(1, CONFIG["STEPS"] // 8) == 0 or step == CONFIG["STEPS"] - 1:
        print(f"step {step:4d}  total={loss.item():7.3f}  sim={parts['sim']:.4f}  "
              f"var={parts['var']:.4f}  cov={parts['cov']:.4f}  emb_std={emb_std:.3f}")

print("Pretraining loop done.")

## The training curves

The four panels tell the story of the run. The total loss and the invariance
(MSE) term should trend down as the predictor gets better at guessing hidden
motion. The variance term should stay low, meaning embeddings keep spreading out,
and the embedding standard deviation should hold well above zero. A healthy run
looks like steady progress with no crash in the embedding spread.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
steps = range(len(history["total"]))
axes[0, 0].plot(steps, history["total"], color="#1d4ed8"); axes[0, 0].set_title("Total loss")
axes[0, 1].plot(steps, history["sim"], color="#ef4444"); axes[0, 1].set_title("Invariance (MSE) term")
axes[1, 0].plot(steps, history["var"], color="#f59e0b"); axes[1, 0].set_title("VICReg variance term")
axes[1, 1].plot(steps, history["emb_std"], color="#22c55e")
axes[1, 1].axhline(CONFIG["VAR_TARGET"], ls="--", color="#94a3b8", label="target std")
axes[1, 1].set_title("Embedding std (collapse monitor)"); axes[1, 1].legend()
for ax in axes.ravel():
    ax.set_xlabel("step"); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print(f"Final embedding std: {history['emb_std'][-1]:.3f} "
      f"(healthy is comfortably above 0; collapse drives it toward 0).")

## Anti-collapse sanity check

One more explicit check. We embed a fresh batch with the trained context encoder
and measure how spread out the per-dimension embeddings are. If the number is
well above zero, the encoder learned varied, informative representations rather
than collapsing to a constant. This is the RQ4 signal the proposal cares about,
and it is the reason VICReg is in the loss at all.

In [ ]:
with torch.no_grad():
    tokens, _ = make_batch(corpus_t, CONFIG, np.random.RandomState(7))
    emb = context_encoder(tokens)                              # (B, N, D)
    per_dim_std = emb.reshape(-1, emb.shape[-1]).std(dim=0)
print(f"Mean per-dimension embedding std: {per_dim_std.mean().item():.3f}")
print(f"Min / max per-dimension std     : {per_dim_std.min().item():.3f} / {per_dim_std.max().item():.3f}")
print("Well above zero means no collapse: the encoder uses many dimensions.")

## Save the trained encoder

We save the context encoder weights along with the config it was built with, so
notebook 05 can rebuild the exact same architecture, load the weights, freeze
them, and turn labeled clips into embeddings. This one file is the whole payoff of
pretraining: a reusable gait encoder learned with no labels.

In [ ]:
ns = CONFIG["CACHE_NS"]
enc_path = CONFIG["CACHE_DIR"] / f"jepa_encoder_gavd{ns}.pt"
enc_path.parent.mkdir(parents=True, exist_ok=True)   # create the cache dir if it does not exist yet

# Stamp the corpus's canonical_id_hash into the encoder config so notebook 05 can
# detect a mismatched encoder (a checkpoint trained on a different locked-id set).
# Falls back to "smoke-or-firstN" for SMOKE or older nb03 caches, per the artifact contract.
encoder_config = {k: v for k, v in CONFIG.items() if k not in ("CACHE_DIR",)}
encoder_config["canonical_id_hash"] = CORPUS_HASH

torch.save({
    "state_dict": context_encoder.state_dict(),
    "config": encoder_config,
}, enc_path)
print(f"Saved trained encoder to {enc_path}")
# The config carries T and N_JOINTS as well as EMBED_DIM and C, because the encoder now
# has time and joint positional embeddings sized to (T, D) and (N_JOINTS, D). Notebook 05
# rebuilds ContextEncoder with these exact values so the weights load and the features match.
print(f"  embed_dim={CONFIG['EMBED_DIM']}, input_dim={CONFIG['C']}, "
      f"T={CONFIG['T']}, n_joints={CONFIG['N_JOINTS']}")
print(f"  canonical_id_hash stamped on encoder: {CORPUS_HASH}")

## Recap and what comes next

We built the four JEPA pieces, tokenized and block-masked the unlabeled corpus,
and ran the pretraining loop. We watched the loss fall while the embedding spread
stayed healthy, confirmed there was no collapse, and saved a reusable encoder,
all without a single label.

In notebook 05 we finally spend the labels. We freeze this encoder, turn each of
the labeled holdout clips into an embedding, and train small probes on the 68-clip
70/30 split. We compare the frozen-probe accuracy against the 76 percent Random
Forest baseline, plot the label-efficiency curve, fit the neuroscience probes,
and run the VICReg on/off ablation to close out the research questions.